<a href="https://colab.research.google.com/github/eryao2023/voice_ai/blob/main/GPT_SoVITS_Web_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

基于[RVC-Boss/GPT-SoVITS/colab_webui.ipynb](https://github.com/RVC-Boss/GPT-SoVITS/blob/main/colab_webui.ipynb)进行了一些完善，便于结合Google Drive更快的进行数据集的上传和训练模型的下载

[官方colab教程链接](https://www.yuque.com/baicaigongchang1145haoyuangong/ib3g1e/zqbopihzr6eqoyl8)

Based on [RVC-Boss/GPT-SoVITS/colab_webui.ipynb](https://github.com/RVC-Boss/GPT-SoVITS/blob/main/colab_webui.ipynb), some improvements have been made to facilitate faster integration with Google Drive Upload the data set and download the training model



---
#### Part0:前置准备 Preparation
---


In [ ]:
#@title 查看CPU/GPU配置

#@markdown Check CPU/GPU configuration
!lscpu | grep -E 'Architecture|Model name|Core\(s\)|CPU\(s\):'
!nvidia-smi

In [2]:
#@title  挂载Google云端硬盘 / Mount Google drive
#@markdown 加载Google云端硬盘（更快地上传数据集文件）

#@markdown Mount Google drive for faster data upload
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
#@title 创建Google Drive数据集上传文件夹
#@markdown Create Google Drive dataset upload folder
rolename = input("请输入本次要训练的角色名称 Please enter the role name for this training：")
!mkdir -p /content/drive/MyDrive/GPT_SoVITS/raw_audio/{rolename}/
print("目录创建成功 Directory created successfully!")
print("请将本次的数据集上传到 Please upload the data set for this training to:")
print("Google Drive：https://drive.google.com/drive/my-drive")
print("/GPT_SoVITS/raw_audio/%s " % rolename)
print("Web UI音频数据集对应目录 Directory of Web UI audio data set:")
print("---------------------------------------------------------")
print("/content/drive/MyDrive/GPT_SoVITS/raw_audio/%s " % rolename)
print("---------------------------------------------------------")

请输入本次要训练的角色名称 Please enter the role name for this training：山田凉
目录创建成功 Directory created successfully!
请将本次的数据集上传到 Please upload the data set for this training to:
Google Drive：https://drive.google.com/drive/my-drive
/GPT_SoVITS/raw_audio/山田凉 
Web UI音频数据集对应目录 Directory of Web UI audio data set:
---------------------------------------------------------
/content/drive/MyDrive/GPT_SoVITS/raw_audio/山田凉 
---------------------------------------------------------


---

#### Part1:环境安装配置 Environment installation configuration
---



In [1]:
#@title STEP:1.1 环境配置下载安装 (≈15min)
!pip install -q condacolab
#@markdown Setting up condacolab and installing packages (≈15min)
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/miniconda/Miniconda3-py39_23.11.0-2-Linux-x86_64.sh")
%cd -q /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS
!conda install -y -q -c pytorch -c nvidia cudatoolkit
%cd -q /content/GPT-SoVITS
!conda install -y -q -c conda-forge gcc gxx ffmpeg cmake -c pytorch -c nvidia
!/usr/local/bin/pip install -r requirements.txt

✨🍰✨ Everything looks OK!
fatal: destination path 'GPT-SoVITS' already exists and is not an empty directory.
Error while loading conda entry point: conda-libmamba-solver (libarchive.so.20: cannot open shared object file: No such file or directory)
/usr/local/lib/python3.9/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(

CondaValueError: You have chosen a non-default solver backend (libmamba) but it was not recognized. Choose one of: classic

Error while loading conda entry point: conda-libmamba-solver (libarchive.so.20: cannot open shared object file: No such 

In [ ]:
#@title STEP1.2 下载预训练模型 (≈2min)
#@markdown Download pretrained models (≈2min)
!mkdir -p /content/GPT-SoVITS/GPT_SoVITS/pretrained_models
!mkdir -p /content/GPT-SoVITS/tools/damo_asr/models
!mkdir -p /content/GPT-SoVITS/tools/uvr5
%cd /content/GPT-SoVITS/GPT_SoVITS/pretrained_models
!git clone https://huggingface.co/lj1995/GPT-SoVITS
%cd /content/GPT-SoVITS/tools/damo_asr/models
!git clone https://www.modelscope.cn/damo/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch.git
!git clone https://www.modelscope.cn/damo/speech_fsmn_vad_zh-cn-16k-common-pytorch.git
!git clone https://www.modelscope.cn/damo/punc_ct-transformer_zh-cn-common-vocab272727-pytorch.git
%cd /content/GPT-SoVITS/tools/uvr5
!git clone https://huggingface.co/Delik/uvr5_weights
!git config core.sparseCheckout true
!mv /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/GPT-SoVITS/* /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/

/content/GPT-SoVITS/GPT_SoVITS/pretrained_models
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 69, done.
remote: Total 69 (delta 0), reused 0 (delta 0), pack-reused 69 (from 1)
Receiving objects: 100% (69/69), 115.56 KiB | 10.50 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Filtering content: 100% (19/19), 4.93 GiB | 29.81 MiB/s, done.
/content/GPT-SoVITS/tools/damo_asr/models
Cloning into 'speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (34/34), done.
error: RPC failed; curl 18 transfer closed with outstanding read data remaining
error: 5797 bytes of body are still expected
fetch-pack: unexpected disconnect while reading sideband packet
fatal: early EOF
fatal: fetch-pack: invalid index-pack output
Cloning into 'speech_fsmn_vad_zh-cn-16k-common-pytorch'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (3


---

#### Part2:模型训练 Model training

---

In [ ]:
#@title WebUI！！ 启动！！！
#@markdown launch WebUI
!/usr/local/bin/pip install ipykernel
!sed -i '10s/False/True/' /content/GPT-SoVITS/config.py
%cd /content/GPT-SoVITS/
!/usr/local/bin/python  webui.py



---

#### Part3:模型文件备份 Model backup (Google Drive)

---



In [ ]:
#@title 备份模型到Google Drive
#@markdown Back up model files to Google Drive

modlename = input("请输入要保存的模型目录名称 Please enter the role name for this training：")

# 创建模型输出目录
!mkdir -p /content/drive/MyDrive/GPT_SoVITS/{modlename}/Model_output/GPT_weights
!mkdir -p /content/drive/MyDrive/GPT_SoVITS/{modlename}/Model_output/SoVITS_weights
print("模型输出目录创建成功！\nThe model output directory is created successfully!")
# 备份模型文件
!cp /content/GPT-SoVITS/GPT_weights/* /content/drive/MyDrive/GPT_SoVITS/{modlename}/Model_output/GPT_weights
!cp /content/GPT-SoVITS/SoVITS_weights/* /content/drive/MyDrive/GPT_SoVITS/{modlename}/Model_output/SoVITS_weights
print("备份成功！Google Drive更新可能有延迟，请多刷新")
print("Backup successful! Google Drive update may be delayed, please refresh frequently.")
print("-------------------------------------")
print("请检查GPT_SoVITS/%s/Model_output/GPT_weights目录下是否存在模型文件！ " % modlename)
print("请检查GPT_SoVITS/%s/Model_output/SoVITS_weights目录下是否存在模型文件！ " % modlename)
print("Please Check GPT_SoVITS/%s/Model_output/GPT_weights " % modlename)
print("Please Check GPT_SoVITS/%s/Model_output/SoVITS_weights " % modlename)
print("Google Drive：https://drive.google.com/drive/my-drive")

In [ ]:
#@title 备份音频切片和标记文件到Google Drive
#@markdown Back up audio slices and tagged files to Google Drive
modlename = input("请输入要保存的模型目录名称 Please enter the role name for this training：")
!mkdir -p /content/drive/MyDrive/GPT_SoVITS/{modlename}/Audio_output/asr_opt
!mkdir -p /content/drive/MyDrive/GPT_SoVITS/{modlename}/Audio_output/slicer_opt
print("模型输出目录创建成功！\nThe model output directory is created successfully!")
!cp /content/GPT-SoVITS/output/asr_opt/* /content/drive/MyDrive/GPT_SoVITS/{modlename}/Audio_output/asr_opt
!cp /content/GPT-SoVITS/output/slicer_opt/* /content/drive/MyDrive/GPT_SoVITS/{modlename}/Audio_output/slicer_opt
print("备份成功！Google Drive更新可能有延迟，请多刷新")
print("Backup successful! Google Drive update may be delayed, please refresh frequently.")
print("-------------------------------------")
print("请检查GPT_SoVITS/%s/Audio_output/asr_opt目录下是否存在音频切片文件！ " % modlename)
print("请检查GPT_SoVITS/%s/Audio_output/slicer_opt目录下是否存在.list标记文件！ " % modlename)
print("Please Check GPT_SoVITS/%s/Audio_output/asr_opt " % modlename)
print("Please Check GPT_SoVITS/%s/Audio_output/slicer_opt " % modlename)
print("Google Drive：https://drive.google.com/drive/my-drive")